# LeetCode 274 — H-Index bằng Python

Link đề: https://leetcode.com/problems/h-index/ · Độ khó: **Medium** · Chủ đề: Array, Counting Sort, Sorting

Skeleton LeetCode:

```python
class Solution:
    def hIndex(self, citations: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (thử mọi `h` và đếm trực tiếp số bài báo có ít nhất `h` trích dẫn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng `citations` (số trích dẫn của từng bài báo), tìm **chỉ số h (h-index)** = giá trị `h` **lớn nhất** sao cho nhà nghiên cứu có **ít nhất `h` bài** mà **mỗi bài** được trích dẫn **ít nhất `h` lần**.
2. Phát biểu lại điều kiện thành dạng dễ kiểm tra: `h` **khả thi** khi **số bài có `citations ≥ h`** là **≥ h**; và nhận ra tính **đơn điệu** (nếu `h` khả thi thì mọi `h' < h` cũng khả thi).
3. Đi từ **thử mọi `h` và đếm (brute force `O(n²)`)** → **sắp xếp giảm dần rồi so `citations[h−1] ≥ h`** (cách tham khảo) → **sắp xếp tăng dần rồi quét** → **đếm theo thùng (counting sort) `O(n)`** → **tìm nhị phân trên `h`**.
4. Hiểu vì sao có thể **chặn trích dẫn ở `n`** (`min(c, n)`) khi đếm thùng: `h` không bao giờ vượt `n`.
5. Tránh các lỗi kinh điển: dùng sai chiều sắp xếp, lệch chỉ số (`h − 1` và `h`), thùng không chặn ở `n` (chỉ số ngoài mảng), quên trường hợp `h = 0`, và nhầm h-index với trích dẫn lớn nhất hay trung vị.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `citations`, trong đó `citations[i]` là số lần bài báo thứ `i` của một nhà nghiên cứu được trích dẫn. Hãy trả về **h-index** của nhà nghiên cứu đó.

Theo định nghĩa của Wikipedia: h-index là giá trị `h` **lớn nhất** sao cho nhà nghiên cứu đã công bố **ít nhất `h` bài báo** mà **mỗi bài được trích dẫn ít nhất `h` lần**.

**Ví dụ 1**

- Input: `citations = [3,0,6,1,5]`
- Output: `3`
- Giải thích: nhà nghiên cứu có `5` bài với số trích dẫn `3, 0, 6, 1, 5`. Có `3` bài được trích dẫn ít nhất `3` lần (`3, 6, 5`) và hai bài còn lại có không quá `3` trích dẫn (`0, 1`). Vậy h-index là `3`.

**Ví dụ 2**

- Input: `citations = [1,3,1]`
- Output: `1`

**Ràng buộc:** `n == citations.length`; `1 <= n <= 5000`; `0 <= citations[i] <= 1000`.

In [1]:
from typing import List
import random
from itertools import accumulate

examples = [
    ([3, 0, 6, 1, 5], 3),
    ([1, 3, 1], 1),
]
for citations, expected in examples:
    print(f"citations={citations}, kỳ vọng {expected}")

citations=[3, 0, 6, 1, 5], kỳ vọng 3
citations=[1, 3, 1], kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Định nghĩa bằng ví dụ: "ít nhất `h` bài, mỗi bài ≥ `h` trích dẫn"

Với `citations = [3, 0, 6, 1, 5]`, thử từng `h`: đếm số bài có `citations ≥ h`; `h` hợp lệ nếu số bài đó `≥ h`.

In [2]:
citations = [3, 0, 6, 1, 5]
for h in range(0, len(citations) + 2):
    count = sum(c >= h for c in citations)
    print(f"h={h}: số bài có >= {h} trích dẫn = {count} -> {'hợp lệ' if count >= h else 'không'}")

h=0: số bài có >= 0 trích dẫn = 5 -> hợp lệ
h=1: số bài có >= 1 trích dẫn = 4 -> hợp lệ
h=2: số bài có >= 2 trích dẫn = 3 -> hợp lệ
h=3: số bài có >= 3 trích dẫn = 3 -> hợp lệ
h=4: số bài có >= 4 trích dẫn = 2 -> không
h=5: số bài có >= 5 trích dẫn = 2 -> không
h=6: số bài có >= 6 trích dẫn = 1 -> không


### 3.2. Tính đơn điệu

Nếu `h` hợp lệ (có ít nhất `h` bài có `≥ h` trích dẫn) thì mọi `h' < h` cũng hợp lệ: các bài đó cũng có `≥ h'` trích dẫn, và số bài `≥ h' ` còn nhiều hơn hoặc bằng `h > h'`. Do đó tập các `h` hợp lệ là một khoảng `0, 1, ..., H`, và **h-index là `H`**, tức bước chuyển từ "hợp lệ" sang "không hợp lệ". Tính chất này cho phép tìm nhị phân.

In [3]:
citations = [3, 0, 6, 1, 5]
valid = [sum(c >= h for c in citations) >= h for h in range(0, 7)]
print(valid)                                   # [True, True, True, True, False, False, False]
assert valid == sorted(valid, reverse=True)    # dạng True...True False...False

[True, True, True, True, False, False, False]


### 3.3. Sắp xếp giảm dần `sort(reverse=True)` và chỉ số `h − 1`

Sau khi sắp **giảm dần**, `citations[0]` là bài nhiều trích dẫn nhất. Bài thứ `h` (chỉ số `h − 1`) có `citations[h − 1]` trích dẫn; nếu `citations[h − 1] ≥ h` thì **`h` bài đầu** đều có `≥ citations[h − 1] ≥ h` trích dẫn: `h` hợp lệ.

In [4]:
citations = sorted([3, 0, 6, 1, 5], reverse=True)
print(citations)                               # [6, 5, 3, 1, 0]
for h in range(1, len(citations) + 1):
    print(f"h={h}: citations[{h - 1}] = {citations[h - 1]} >= {h}? {citations[h - 1] >= h}")

[6, 5, 3, 1, 0]
h=1: citations[0] = 6 >= 1? True
h=2: citations[1] = 5 >= 2? True
h=3: citations[2] = 3 >= 3? True
h=4: citations[3] = 1 >= 4? False
h=5: citations[4] = 0 >= 5? False


### 3.4. Đếm theo thùng (counting sort) và tổng tích lũy

Khi các giá trị là số nguyên nhỏ, ta không cần sắp xếp: dùng mảng `bucket` với `bucket[v]` là số phần tử bằng `v`. Tổng tích lũy từ cuối về đầu cho "số phần tử `≥ v`". Giá trị `v` lớn hơn `n` đều có tác dụng như `n` (vì `h ≤ n`), nên ta **chặn** bằng `min(c, n)`.

In [5]:
citations = [3, 0, 6, 1, 5]
n = len(citations)
bucket = [0] * (n + 1)
for c in citations:
    bucket[min(c, n)] += 1                     # chặn ở n
print("bucket =", bucket)                      # số bài có trích dẫn = v (v = 5 gồm cả 6)
total = 0
for v in range(n, -1, -1):
    total += bucket[v]                         # total = số bài có >= v trích dẫn
    print(f"v={v}: số bài >= {v} = {total}")

bucket = [1, 1, 0, 1, 0, 2]
v=5: số bài >= 5 = 2
v=4: số bài >= 4 = 2
v=3: số bài >= 3 = 3
v=2: số bài >= 2 = 3
v=1: số bài >= 1 = 4
v=0: số bài >= 0 = 5


### 3.5. Tìm nhị phân trên đáp án

Vì tập `h` hợp lệ có dạng `0..H`, ta tìm `H` bằng nhị phân: kiểm tra `mid` bằng cách đếm số bài `≥ mid` (tốn `O(n)`); nếu hợp lệ thì `lo = mid`, ngược lại `hi = mid − 1`. Dùng `mid = (lo + hi + 1) // 2` để không bị lặp vô hạn khi `lo = mid`.

In [6]:
def feasible(citations, h):
    return sum(c >= h for c in citations) >= h


citations = [3, 0, 6, 1, 5]
lo, hi = 0, len(citations)
while lo < hi:
    mid = (lo + hi + 1) // 2
    print(f"lo={lo}, hi={hi}, mid={mid}, khả thi? {feasible(citations, mid)}")
    if feasible(citations, mid):
        lo = mid
    else:
        hi = mid - 1
print("h-index =", lo)

lo=0, hi=5, mid=3, khả thi? True
lo=3, hi=5, mid=4, khả thi? False
h-index = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `[3, 0, 6, 1, 5]` (`n = 5`). Sắp giảm dần: `[6, 5, 3, 1, 0]`; thứ hạng bài `1, 2, 3, 4, 5`.

| hạng `h` | trích dẫn của bài hạng `h` | `≥ h`? | ý nghĩa |
|---|---|---|---|
| 1 | 6 | có | 1 bài có ≥ 1 trích dẫn |
| 2 | 5 | có | 2 bài có ≥ 2 |
| 3 | 3 | có | 3 bài có ≥ 3 |
| 4 | 1 | không | không đủ 4 bài có ≥ 4 |
| 5 | 0 | không | không đủ 5 bài có ≥ 5 |

`h` lớn nhất thỏa là `3`. Nhìn cách khác: **số trích dẫn giảm dần theo hạng, còn hạng thì tăng dần**; h-index là chỗ hai đường "gặp nhau": bài hạng `h` vẫn còn `≥ h` trích dẫn nhưng bài hạng `h + 1` thì không.

In [7]:
ranked = sorted([3, 0, 6, 1, 5], reverse=True)
best = 0
for rank, c in enumerate(ranked, start=1):
    ok = c >= rank
    print(f"hạng {rank}: {c} trích dẫn {'>=' if ok else '<'} {rank}")
    if ok:
        best = rank
print("h-index =", best)

hạng 1: 6 trích dẫn >= 1
hạng 2: 5 trích dẫn >= 2
hạng 3: 3 trích dẫn >= 3
hạng 4: 1 trích dẫn < 4
hạng 5: 0 trích dẫn < 5
h-index = 3


## 5. Cách 1 — Brute Force / baseline: thử mọi `h` và đếm

**Ý tưởng.** Thử `h` từ `n` giảm dần về `0`; với mỗi `h` đếm số bài có `citations ≥ h` bằng một vòng lặp; `h` đầu tiên có đếm `≥ h` là đáp án. (`h = 0` luôn hợp lệ nên vòng lặp luôn kết thúc.)

In [8]:
def hindex_bruteforce(citations: list[int]) -> int:
    n = len(citations)
    for h in range(n, -1, -1):              # thử h lớn nhất trước
        count = 0
        for c in citations:
            if c >= h:
                count += 1
        if count >= h:                      # có ít nhất h bài có >= h trích dẫn
            return h
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex_bruteforce(citations)} (kỳ vọng {expected})")

citations=[3, 0, 6, 1, 5] -> 3 (kỳ vọng 3)
citations=[1, 3, 1] -> 1 (kỳ vọng 1)


**Độ phức tạp (Cách 1).** Gọi `n = len(citations)`. Tối đa `n + 1` giá trị `h`, mỗi giá trị đếm `O(n)`: **Time `O(n²)`**, **Space `O(1)`**. Với `n = 5000` là `2.5·10⁷` phép so sánh: vẫn qua được nhưng chưa tốt.

**Nút thắt.** Mỗi `h` đếm lại từ đầu. Nếu mảng đã được sắp xếp thì "số bài có `≥ h` trích dẫn" tra cứu ngay theo vị trí; hoặc ta đếm trước một lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp giảm dần rồi so với hạng.** Sau khi sắp `citations` giảm dần, "số bài có `≥ h` trích dẫn" là số phần tử đầu mảng có giá trị `≥ h`. Bài hạng `h` (`citations[h − 1]`) quyết định: nếu `citations[h − 1] ≥ h` thì `h` bài đầu đều `≥ h` (vì giảm dần), nên `h` hợp lệ. Ngược lại `h` không hợp lệ (bài hạng `h` quá ít trích dẫn, mà bài khác hạng thấp hơn còn ít hơn nữa).

Vì `h` hợp lệ có tính đơn điệu, ta thử `h` từ `n` xuống `1`; `h` đầu tiên có `citations[h − 1] ≥ h` là đáp án; nếu không có thì `0`.

**Không sắp xếp:** vì `h ≤ n`, trích dẫn lớn hơn `n` chỉ cần "tính là `n`". Ta đếm số bài theo giá trị đã chặn trong mảng thùng `bucket[0..n]` rồi cộng dồn từ `n` xuống để biết "số bài `≥ h`" cho mọi `h` trong `O(n)`.

In [9]:
citations = [3, 0, 6, 1, 5]
n = len(citations)
ranked = sorted(citations, reverse=True)
# h-index là số hạng h lớn nhất thỏa ranked[h - 1] >= h
print("h-index (sắp xếp):", max([h for h in range(1, n + 1) if ranked[h - 1] >= h], default=0))
bucket = [0] * (n + 1)
for c in citations:
    bucket[min(c, n)] += 1
total = 0
for h in range(n, -1, -1):
    total += bucket[h]
    if total >= h:
        print("h-index (thùng):", h)
        break

h-index (sắp xếp): 3
h-index (thùng): 3


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp giảm dần (cách của lời giải tham khảo)

**Giải thích.** `citations.sort(reverse=True)`; duyệt `h` từ `n` về `1`; trả `h` đầu tiên có `citations[h − 1] ≥ h`; nếu không có thì `0`. Ta sắp trên **bản sao** để không sửa đầu vào trong phần test.

In [10]:
def hindex_sort_desc(citations: list[int]) -> int:
    c = sorted(citations, reverse=True)           # giảm dần
    for h in range(len(c), 0, -1):
        if c[h - 1] >= h:                          # bài hạng h có ít nhất h trích dẫn
            return h
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex_sort_desc(citations)} (kỳ vọng {expected})")

citations=[3, 0, 6, 1, 5] -> 3 (kỳ vọng 3)
citations=[1, 3, 1] -> 1 (kỳ vọng 1)


In [11]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Solution:
    def hIndex(self, citations: List[int]) -> int:
        citations.sort(reverse=True)
        for h in range(len(citations), 0, -1):
            if citations[h - 1] >= h:
                return h
        return 0

**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** nếu sắp trên bản sao (hoặc `O(log n)` khi sắp tại chỗ).

### Cách 3 — Sắp xếp tăng dần rồi quét

**Giải thích.** Sắp **tăng dần** `c`. Với chỉ số `i` (từ `0`), có `n − i` bài từ vị trí `i` trở đi, đều có `≥ c[i]` trích dẫn. Nếu `c[i] ≥ n − i` thì `h = n − i` hợp lệ. Quét `i` từ `0` tăng dần: `h = n − i` giảm dần, nên `i` **đầu tiên** thỏa cho `h` lớn nhất: `h = n − i`. Nếu không có `i` nào thỏa thì `0`.

In [12]:
def hindex_sort_asc(citations: list[int]) -> int:
    c = sorted(citations)                          # tăng dần
    n = len(c)
    for i in range(n):
        if c[i] >= n - i:                          # n - i bài (từ i đến cuối) đều >= c[i] >= n - i
            return n - i
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex_sort_asc(citations)} (kỳ vọng {expected})")

citations=[3, 0, 6, 1, 5] -> 3 (kỳ vọng 3)
citations=[1, 3, 1] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 3:** **Time `O(n log n)`**, **Space `O(n)`**. Chính là công thức của bài **H-Index II** (mảng đã sắp tăng): `citations[n − h] ≥ h`.

### Cách 4 — Đếm theo thùng (counting sort), `O(n)`

**Giải thích.** `bucket` có `n + 1` ô; với mỗi `c` tăng `bucket[min(c, n)]`. Cộng dồn `total` từ `h = n` xuống `0`: `total` là số bài có `≥ h` trích dẫn (vì các bài bị chặn về `n` vẫn `≥ h` với mọi `h ≤ n`). `h` đầu tiên (lớn nhất) có `total ≥ h` là đáp án.

In [13]:
def hindex_counting(citations: list[int]) -> int:
    n = len(citations)
    bucket = [0] * (n + 1)
    for c in citations:
        bucket[min(c, n)] += 1                     # chặn trích dẫn ở n
    total = 0                                      # số bài có >= h trích dẫn
    for h in range(n, -1, -1):
        total += bucket[h]
        if total >= h:
            return h
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex_counting(citations)} (kỳ vọng {expected})")

citations=[3, 0, 6, 1, 5] -> 3 (kỳ vọng 3)
citations=[1, 3, 1] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 4:** **Time `O(n)`** (một lượt đếm, một lượt cộng dồn), **Space `O(n)`** cho `bucket`. Vì `citations[i] ≤ 1000` ta cũng có thể dùng thùng cỡ `1001` (không chặn), nhưng chặn ở `n` tổng quát hơn.

### Cách 5 — Tìm nhị phân trên `h` với kiểm tra bằng đếm

**Giải thích.** Dùng tính đơn điệu: `lo = 0`, `hi = n`; `mid = (lo + hi + 1) // 2`; nếu `sum(c ≥ mid) ≥ mid` thì `lo = mid` ngược lại `hi = mid − 1`. Mỗi lần kiểm tra `O(n)`, tổng `O(n log n)`; không cần sắp xếp.

In [14]:
def hindex_binary_search(citations: list[int]) -> int:
    lo, hi = 0, len(citations)
    while lo < hi:
        mid = (lo + hi + 1) // 2                   # lấy cận trên để tránh lặp vô hạn
        if sum(c >= mid for c in citations) >= mid:
            lo = mid                                # mid hợp lệ: thử lớn hơn
        else:
            hi = mid - 1                            # mid không hợp lệ
    return lo


for citations, expected in examples:
    print(f"citations={citations} -> {hindex_binary_search(citations)} (kỳ vọng {expected})")

citations=[3, 0, 6, 1, 5] -> 3 (kỳ vọng 3)
citations=[1, 3, 1] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 5:** **Time `O(n log n)`**, **Space `O(1)`**. Hợp khi không muốn sắp xếp hoặc dùng thêm bộ nhớ.

> Cách 2 là cách tham khảo; Cách 4 là tối ưu theo thời gian (`O(n)`); Cách 3 dẫn tới H-Index II; Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 4 (đếm thùng) trên `[3, 0, 6, 1, 5]` và Cách 2 (sắp xếp giảm dần) trên `[1, 3, 1]`.

In [15]:
def trace_counting(citations):
    n = len(citations)
    bucket = [0] * (n + 1)
    print(f"citations={citations}, n={n}")
    for c in citations:
        bucket[min(c, n)] += 1
        print(f"  c={c} -> chặn thành {min(c, n)}: bucket={bucket}")
    total = 0
    for h in range(n, -1, -1):
        total += bucket[h]
        print(f"  h={h}: total += bucket[{h}]={bucket[h]} -> total={total} {'>=' if total >= h else '<'} {h}")
        if total >= h:
            print("  => h-index =", h)
            return h


assert trace_counting([3, 0, 6, 1, 5]) == 3

citations=[3, 0, 6, 1, 5], n=5
  c=3 -> chặn thành 3: bucket=[0, 0, 0, 1, 0, 0]
  c=0 -> chặn thành 0: bucket=[1, 0, 0, 1, 0, 0]
  c=6 -> chặn thành 5: bucket=[1, 0, 0, 1, 0, 1]
  c=1 -> chặn thành 1: bucket=[1, 1, 0, 1, 0, 1]
  c=5 -> chặn thành 5: bucket=[1, 1, 0, 1, 0, 2]
  h=5: total += bucket[5]=2 -> total=2 < 5
  h=4: total += bucket[4]=0 -> total=2 < 4
  h=3: total += bucket[3]=1 -> total=3 >= 3
  => h-index = 3


In [16]:
def trace_sort_desc(citations):
    c = sorted(citations, reverse=True)
    print(f"citations={citations}, sắp giảm dần = {c}")
    for h in range(len(c), 0, -1):
        ok = c[h - 1] >= h
        print(f"  h={h}: c[{h - 1}]={c[h - 1]} {'>=' if ok else '<'} {h}")
        if ok:
            print("  => h-index =", h)
            return h
    print("  => h-index = 0")
    return 0


assert trace_sort_desc([1, 3, 1]) == 1
print()
assert trace_sort_desc([0, 0, 0]) == 0

citations=[1, 3, 1], sắp giảm dần = [3, 1, 1]
  h=3: c[2]=1 < 3
  h=2: c[1]=1 < 2
  h=1: c[0]=3 >= 1
  => h-index = 1

citations=[0, 0, 0], sắp giảm dần = [0, 0, 0]
  h=3: c[2]=0 < 3
  h=2: c[1]=0 < 2
  h=1: c[0]=0 < 1
  => h-index = 0


Bảng tóm tắt (Cách 4, `[3, 0, 6, 1, 5]`): sau khi chặn `6 → 5`, `bucket = [1, 1, 0, 1, 0, 2]`.

| `h` | `bucket[h]` | `total` (số bài `≥ h`) | `total ≥ h`? |
|---|---|---|---|
| 5 | 2 | 2 | không |
| 4 | 0 | 2 | không |
| 3 | 1 | 3 | **có** → đáp án `3` |

Vì ta đi từ `h` lớn xuống nhỏ nên `h` đầu tiên thỏa chính là lớn nhất.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (đếm theo thùng, `O(n)`) cho tốc độ tốt nhất; không sửa mảng đầu vào. Đúng class `Solution`, đúng method `hIndex(self, citations: List[int]) -> int`.

In [17]:
from typing import List


class Solution:
    def hIndex(self, citations: List[int]) -> int:
        n = len(citations)
        bucket = [0] * (n + 1)
        for c in citations:
            bucket[min(c, n)] += 1               # h không vượt n nên chặn trích dẫn ở n
        total = 0                                # số bài có ít nhất h trích dẫn
        for h in range(n, -1, -1):               # từ lớn xuống nhỏ: h đầu tiên thỏa là lớn nhất
            total += bucket[h]
            if total >= h:
                return h
        return 0


for citations, expected in examples:
    assert Solution().hIndex(citations) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Đơn điệu:** gọi `N(h)` là số bài có `citations ≥ h`. `N` không tăng theo `h`, nên điều kiện `N(h) ≥ h` (vế trái không tăng, vế phải tăng) đúng cho một đoạn `0..H` rồi sai mãi: `H` là điểm chuyển, tức h-index.
- **Cách 2 (sắp giảm dần):** sau sắp, `N(h) ≥ h ⇔ citations[h − 1] ≥ h`: nếu `c[h − 1] ≥ h` thì các phần tử `c[0..h−1] ≥ c[h − 1] ≥ h`, tức `N(h) ≥ h`; ngược lại nếu `c[h − 1] < h` thì chỉ có tối đa `h − 1` phần tử (các phần tử đầu) `≥ h`: `N(h) ≤ h − 1`.
- **Cách 3 (sắp tăng):** `N(h) ≥ h ⇔ c[n − h] ≥ h` (phần tử thứ `h` từ cuối); đặt `i = n − h`.
- **Cách 4 (thùng):** với `h ≤ n`, `c ≥ h ⇔ min(c, n) ≥ h`, nên chặn ở `n` không đổi `N(h)`. Cộng dồn `total` từ `n` xuống `h` đúng bằng `N(h)`. Trả `h` đầu tiên có `total ≥ h` (lớn nhất). Vì `N(0) = n ≥ 0`, luôn tồn tại đáp án ≥ 0.
- **Cách 5:** nhị phân trên tính đơn điệu; `mid = (lo + hi + 1) // 2` đảm bảo khoảng thu hẹp.
- **Kết quả không vượt `n`** (vì cần `h` bài khác nhau) và không vượt trích dẫn lớn nhất.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(citations)`.

- **Cách 1 (thử mọi `h` và đếm):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (sắp xếp giảm dần):** Time `O(n log n)`, Space `O(n)` (hoặc `O(log n)` sắp tại chỗ).
- **Cách 3 (sắp xếp tăng dần):** Time `O(n log n)`, Space `O(n)`.
- **Cách 4 (đếm thùng):** Time `O(n)`, Space `O(n)`.
- **Cách 5 (nhị phân trên `h`):** Time `O(n log n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất như nhau cho mọi cách. Với `n ≤ 5000`, mọi cách đều chạy nhanh.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` (mỗi lần truyền bản sao vì Cách 2 nộp bài sắp tại chỗ). `oracle` thử mọi `h` từ `0` đến `n + 1` bằng cách đếm trực tiếp rồi lấy `h` lớn nhất thỏa điều kiện — không dùng sắp xếp, thùng hay nhị phân.

In [18]:
sol = Solution()
impls = {
    "bruteforce": hindex_bruteforce,
    "sort_desc": hindex_sort_desc,
    "sort_asc": hindex_sort_asc,
    "counting": hindex_counting,
    "binary_search": hindex_binary_search,
    "submit": sol.hIndex,
    "submit_sort_inplace": lambda c: Solution2().hIndex(c),
}


class Solution2:
    """Lời giải tham khảo sắp tại chỗ (để thử cả cách nộp thứ hai)."""
    def hIndex(self, citations):
        citations.sort(reverse=True)
        for h in range(len(citations), 0, -1):
            if citations[h - 1] >= h:
                return h
        return 0


def oracle(citations):
    best = 0
    for h in range(0, len(citations) + 2):
        count = sum(1 for c in citations if c >= h)
        if count >= h:
            best = h
    return best


def run_all(citations):
    out = {}
    for name, fn in impls.items():
        original = list(citations)
        out[name] = fn(original)
        if name != "submit_sort_inplace":
            assert original == citations, (name, "đã sửa đầu vào")
    return out


for citations, expected in examples:
    assert oracle(citations) == expected
    for name, got in run_all(citations).items():
        assert got == expected, (name, citations, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(citations, expected, label):
    assert oracle(citations) == expected, (label, "oracle", oracle(citations), expected)
    for name, got in run_all(citations).items():
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([0], 0, "n = 1, không có trích dẫn")
check_all([1], 1, "n = 1, một trích dẫn")
check_all([1000], 1, "n = 1, trích dẫn lớn: h-index vẫn 1")
check_all([0, 0, 0, 0], 0, "toàn 0")
check_all([1, 1, 1, 1], 1, "toàn 1")
check_all([4, 4, 4, 4], 4, "toàn n: h = n")
check_all([100, 100, 100], 3, "trích dẫn rất lớn: h = n")
check_all([3, 0, 6, 1, 5], 3, "ví dụ 1")
check_all([1, 3, 1], 1, "ví dụ 2")
check_all([2, 2], 2, "n = 2, h = 2")
check_all([5, 0], 1, "một bài lớn, một bài 0")
check_all([0, 1], 1, "đảo thứ tự")
check_all([10, 8, 5, 4, 3], 4, "[10,8,5,4,3]: h = 4")
check_all([1000] * 5000, 1000, "n = 5000, mọi bài 1000: h = 1000")
check_all(list(range(5000)), 2500, "n = 5000, trích dẫn 0..4999 (chặn ở 1000 theo đề nhưng test hợp lệ cho thuật toán)")

PASS n = 1, không có trích dẫn -> 0
PASS n = 1, một trích dẫn -> 1
PASS n = 1, trích dẫn lớn: h-index vẫn 1 -> 1
PASS toàn 0 -> 0
PASS toàn 1 -> 1
PASS toàn n: h = n -> 4
PASS trích dẫn rất lớn: h = n -> 3
PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 1
PASS n = 2, h = 2 -> 2
PASS một bài lớn, một bài 0 -> 1
PASS đảo thứ tự -> 1
PASS [10,8,5,4,3]: h = 4 -> 4


PASS n = 5000, mọi bài 1000: h = 1000 -> 1000


PASS n = 5000, trích dẫn 0..4999 (chặn ở 1000 theo đề nhưng test hợp lệ cho thuật toán) -> 2500


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(274)

# Tính chất 1: 0 <= h <= min(n, max(citations)); N(h) >= h và N(h + 1) < h + 1
for _ in range(500):
    n = random.randint(1, 30)
    citations = [random.randint(0, 40) for _ in range(n)]
    h = sol.hIndex(citations)
    assert 0 <= h <= min(n, max(citations))
    assert sum(c >= h for c in citations) >= h
    assert sum(c >= h + 1 for c in citations) < h + 1
print("PASS tính chất: N(h) >= h và N(h + 1) < h + 1")

# Tính chất 2: hoán vị không đổi; thêm một bài báo không làm h-index giảm; tăng trích dẫn không làm h giảm
for _ in range(300):
    n = random.randint(1, 20)
    citations = [random.randint(0, 25) for _ in range(n)]
    shuffled = citations[:]
    random.shuffle(shuffled)
    h = sol.hIndex(citations)
    assert sol.hIndex(shuffled) == h
    assert sol.hIndex(citations + [random.randint(0, 25)]) >= h
    bumped = citations[:]
    bumped[random.randrange(n)] += random.randint(0, 5)
    assert sol.hIndex(bumped) >= h
print("PASS tính chất: hoán vị, thêm bài và tăng trích dẫn không làm h-index giảm")

# Phản ví dụ: h-index không phải trích dẫn lớn nhất hay trung vị
c = [10, 8, 5, 4, 3]
print("max =", max(c), "| trung vị =", sorted(c)[len(c) // 2], "| h-index =", sol.hIndex(c))
assert sol.hIndex(c) == 4 and max(c) == 10 and sorted(c)[len(c) // 2] == 5

PASS tính chất: N(h) >= h và N(h + 1) < h + 1
PASS tính chất: hoán vị, thêm bài và tăng trích dẫn không làm h-index giảm
max = 10 | trung vị = 5 | h-index = 4


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(1, 25)
    hi = random.choice([1, 3, 10, 30, 1000])
    citations = [random.randint(0, hi) for _ in range(n)]
    expected = oracle(citations)
    for name, got in run_all(citations).items():
        assert got == expected, (name, citations, got, expected)

for trial in range(40):                          # n tới 5000, trích dẫn <= 1000 (theo đề)
    n = random.randint(1000, 5000)
    citations = [random.randint(0, random.choice([5, 50, 1000])) for _ in range(n)]
    expected = hindex_counting(citations)
    for name, fn in impls.items():
        if name in ("bruteforce",):
            continue
        assert fn(list(citations)) == expected, (name, n)
    assert oracle(citations) == expected
print("PASS 3000 input ngẫu nhiên khớp oracle độc lập; 40 mảng dài (n tới 5000) khớp giữa các cách")

PASS 3000 input ngẫu nhiên khớp oracle độc lập; 40 mảng dài (n tới 5000) khớp giữa các cách


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp tăng dần nhưng dùng công thức của sắp xếp giảm dần

In [22]:
def sai_sai_chieu(citations):
    c = sorted(citations)                          # tăng dần
    for h in range(len(c), 0, -1):
        if c[h - 1] >= h:                          # sai: công thức này chỉ đúng khi sắp GIẢM dần
            return h
    return 0


print("[3,0,6,1,5] -> sai:", sai_sai_chieu([3, 0, 6, 1, 5]), "| đúng:", Solution().hIndex([3, 0, 6, 1, 5]))

[3,0,6,1,5] -> sai: 5 | đúng: 3


**Vì sao sai:** `c[h − 1] ≥ h` mang nghĩa "bài **hạng `h`**" chỉ khi mảng sắp **giảm dần**. Sắp tăng thì `c[h − 1]` là bài nhỏ thứ `h`, không còn ý nghĩa hạng. Với mảng tăng, công thức đúng là `c[n − h] ≥ h`.

### Lỗi 2: Thùng không chặn ở `n` (chỉ số ngoài mảng)

In [23]:
def sai_khong_chan(citations):
    n = len(citations)
    bucket = [0] * (n + 1)
    for c in citations:
        bucket[c] += 1                            # sai: c có thể lớn hơn n
    total = 0
    for h in range(n, -1, -1):
        total += bucket[h]
        if total >= h:
            return h
    return 0


try:
    sai_khong_chan([3, 0, 6, 1, 5])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().hIndex([3, 0, 6, 1, 5]))

IndexError: list index out of range
đúng: 3


**Vì sao sai:** trích dẫn `6 > n = 5` làm chỉ số `bucket[6]` ngoài mảng (`IndexError`). Vì `h ≤ n`, mọi `c ≥ n` đều "tính như `n`": chặn bằng `min(c, n)`.

### Lỗi 3: Lệch chỉ số: trả `h − 1` hoặc dùng `c[h]` thay cho `c[h − 1]`

In [24]:
def sai_lech_chi_so(citations):
    c = sorted(citations, reverse=True)
    for h in range(len(c) - 1, 0, -1):               # sai: bỏ h = n
        if c[h - 1] >= h:
            return h
    return 0


print("[4,4,4,4] -> sai:", sai_lech_chi_so([4, 4, 4, 4]), "| đúng:", Solution().hIndex([4, 4, 4, 4]))

[4,4,4,4] -> sai: 3 | đúng: 4


**Vì sao sai:** `h` có thể bằng `n` (khi **mọi** bài có `≥ n` trích dẫn). Vòng lặp phải bắt đầu từ `h = n` (`range(n, 0, -1)`). Với chỉ số, `c[h − 1]` là bài hạng `h` (đếm từ 1).

### Lỗi 4: Quên trường hợp `h = 0`

In [25]:
def sai_quen_khong(citations):
    c = sorted(citations, reverse=True)
    ans = 1                                          # sai: mặc định 1
    for h in range(len(c), 0, -1):
        if c[h - 1] >= h:
            return h
    return ans


print("[0,0,0] -> sai:", sai_quen_khong([0, 0, 0]), "| đúng:", Solution().hIndex([0, 0, 0]))

[0,0,0] -> sai: 1 | đúng: 0


**Vì sao sai:** nếu mọi bài đều `0` trích dẫn thì không `h ≥ 1` nào thỏa; h-index là `0`. Giá trị mặc định khi không tìm thấy phải là `0`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Tìm `h` lớn nhất sao cho có ít nhất `h` phần tử `≥ h`" (như h-index): **sắp xếp** rồi so với thứ hạng, hoặc **đếm thùng** chặn ở `n`.
- Điều kiện có dạng đúng cho một khoảng `0..H` rồi sai: **tìm nhị phân trên đáp án**.
- Khi các giá trị là số nguyên nhỏ hoặc đáp án bị chặn trên: **counting sort** cho `O(n)`.
- Chặn giá trị ở cận của đáp án (`min(c, n)`) để thùng không vượt kích thước.
- Mảng đã **sắp tăng** (H-Index II): `c[n − h] ≥ h` và tìm nhị phân `O(log n)`.
- Test: oracle đếm trực tiếp với mọi `h`, rồi kiểm tra `N(h) ≥ h` và `N(h + 1) < h + 1`.

## 15. Biến thể của bài

1. **H-Index II (LeetCode 275):** mảng đã sắp tăng, yêu cầu `O(log n)` (bài kế tiếp).
2. **g-index, i10-index:** các chỉ số khác của nhà nghiên cứu.
3. **h-index sau khi thêm `k` trích dẫn** tùy chọn vào các bài.
4. **Số bài ít nhất cần thêm một trích dẫn để h-index tăng 1.**
5. **h-index trên luồng dữ liệu** (thêm bài lần lượt): dùng thùng cộng dồn.

In [26]:
def i10_index(citations):
    """Biến thể 2: số bài có ít nhất 10 trích dẫn."""
    return sum(c >= 10 for c in citations)


def g_index(citations):
    """Biến thể 2: g lớn nhất sao cho g bài đầu (giảm dần) có tổng trích dẫn >= g^2."""
    c = sorted(citations, reverse=True)
    best, total = 0, 0
    for g, x in enumerate(c, start=1):
        total += x
        if total >= g * g:
            best = g
    return best


def hindex_after_adding(citations, k):
    """Biến thể 3: có thể cộng tổng cộng k trích dẫn tùy ý vào các bài; h-index lớn nhất đạt được (thử mọi h)."""
    c = sorted(citations, reverse=True)
    n = len(c)
    best = 0
    for h in range(1, n + 1):
        need = sum(max(0, h - x) for x in c[:h])       # cộng cho h bài đầu để đều >= h
        if need <= k:
            best = h
    return best


class HIndexStream:
    """Biến thể 5: thêm bài lần lượt, h-index tính nhanh bằng thùng (cập nhật theo giá trị)."""
    def __init__(self):
        self.items = []

    def add(self, c):
        self.items.append(c)
        return Solution().hIndex(self.items)


assert i10_index([10, 9, 11, 3]) == 2
assert g_index([3, 0, 6, 1, 5]) == 3 and g_index([1, 1, 1]) == 1
assert hindex_after_adding([3, 0, 6, 1, 5], 0) == 3
assert hindex_after_adding([3, 0, 6, 1, 5], 3) == 3           # đạt h = 4 cần 1 + 3 = 4 trích dẫn thêm
assert hindex_after_adding([3, 0, 6, 1, 5], 4) == 4
stream = HIndexStream()
assert [stream.add(c) for c in [3, 0, 6, 1, 5]] == [1, 1, 2, 2, 3]
random.seed(15)
for _ in range(200):
    c = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert hindex_after_adding(c, 0) == Solution().hIndex(c)
print("PASS biến thể: i10, g-index, thêm trích dẫn, luồng")

PASS biến thể: i10, g-index, thêm trích dẫn, luồng


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `h` và đếm | đếm bài `≥ h` cho từng `h` | `O(n²)` | `O(1)` | hiểu đề; oracle |
| 2. Sắp giảm dần | `c[h − 1] ≥ h` | `O(n log n)` | `O(n)` | cách tham khảo, dễ nhớ |
| 3. Sắp tăng dần | `c[i] ≥ n − i` | `O(n log n)` | `O(n)` | dẫn tới H-Index II |
| 4. Đếm thùng | `bucket[min(c, n)]` + cộng dồn | **`O(n)`** | `O(n)` | **cách nộp bài** (nhanh nhất) |
| 5. Nhị phân trên `h` | điều kiện đơn điệu | `O(n log n)` | `O(1)` | không muốn sắp xếp / thêm bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([2, 7, 1], reverse=True)`, `min(9, 5)`, `Solution().hIndex([0])`, `Solution().hIndex([5, 5, 5])`, `Solution().hIndex([1, 1, 3, 3])`.

**Bài 2 (tính tay).** Với `citations = [6, 5, 3, 1, 0]` và `[10, 8, 5, 4, 3]` và `[1, 2, 2, 2]`: sắp giảm dần, lập bảng hạng `h` và `c[h − 1] ≥ h`, tìm h-index.

**Bài 3 (điền TODO).** Hoàn thành `hindex_todo(citations)` theo Cách 4 (đếm thùng).

In [27]:
def hindex_todo(citations):
    n = len(citations)
    # TODO 1: bucket = [0] * (n + 1); với mỗi c tăng bucket[min(c, n)]
    # TODO 2: total = 0; duyệt h từ n về 0: total += bucket[h]
    # TODO 3: nếu total >= h thì trả về h
    # TODO 4: (cuối cùng) trả về 0
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `hindex_asc2(citations)` theo Cách 3 (sắp tăng, `c[i] ≥ n − i`) mà không nhìn code mẫu. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `papers_to_cite_for_next(citations)` trả về **số trích dẫn tối thiểu cần thêm** (cộng tùy ý vào các bài) để h-index tăng thêm 1. (b) Viết `hindex_sorted_asc(citations)` cho mảng **đã sắp tăng** trong `O(log n)` (tìm nhị phân trên `c[n − h] ≥ h`) — bài H-Index II.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `sorted(reverse=True)` sắp giảm dần; h-index không vượt số bài. Bài 2: bài hạng `h` có ít nhất `h` trích dẫn.
- **Mức 2 (vừa):** Bài 1: `[7, 2, 1]`, `5`, `0`, `3`, `2`. Bài 2: `[6,5,3,1,0]` → 3; `[10,8,5,4,3]`: `c[3] = 4 ≥ 4` nên `4`, `c[4] = 3 < 5`: h = 4; `[1,2,2,2]`: sắp `[2,2,2,1]`: `c[1] = 2 ≥ 2`, `c[2] = 2 < 3`: h = 2. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `for i in range(n): if c[i] >= n - i: return n - i`. Bài 5(a): với `h = hiện tại + 1`, cộng cho `h` bài đầu (giảm dần) lượng `max(0, h − c)`. Bài 5(b): `lo, hi = 0, n`; `mid = (lo + hi + 1) // 2`; nếu `c[n − mid] ≥ mid` thì `lo = mid` ngược lại `hi = mid − 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert sorted([2, 7, 1], reverse=True) == [7, 2, 1] and min(9, 5) == 5
assert Solution().hIndex([0]) == 0 and Solution().hIndex([5, 5, 5]) == 3 and Solution().hIndex([1, 1, 3, 3]) == 2

# Bài 2
assert Solution().hIndex([6, 5, 3, 1, 0]) == 3
assert Solution().hIndex([10, 8, 5, 4, 3]) == 4
assert Solution().hIndex([1, 2, 2, 2]) == 2

# Bài 3
def hindex_todo_ans(citations):
    n = len(citations)
    bucket = [0] * (n + 1)
    for c in citations:
        bucket[min(c, n)] += 1
    total = 0
    for h in range(n, -1, -1):
        total += bucket[h]
        if total >= h:
            return h
    return 0

for citations, expected in examples:
    assert hindex_todo_ans(citations) == expected

# Bài 4
def hindex_asc2(citations):
    c = sorted(citations)
    n = len(c)
    for i in range(n):
        if c[i] >= n - i:
            return n - i
    return 0

for citations, expected in examples:
    assert hindex_asc2(citations) == expected

# Bài 5(a), 5(b)
def papers_to_cite_for_next(citations):
    c = sorted(citations, reverse=True)
    h = Solution().hIndex(citations) + 1
    if h > len(c):
        return None                                  # không thể tăng h-index (không đủ bài)
    return sum(max(0, h - x) for x in c[:h])

def hindex_sorted_asc(citations):
    n = len(citations)
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if citations[n - mid] >= mid:
            lo = mid
        else:
            hi = mid - 1
    return lo

assert papers_to_cite_for_next([3, 0, 6, 1, 5]) == 4
assert papers_to_cite_for_next([4, 4, 4, 4]) is None
random.seed(19)
for _ in range(300):
    c = sorted(random.randint(0, 20) for _ in range(random.randint(1, 15)))
    assert hindex_sorted_asc(c) == Solution().hIndex(c[:])
    nxt = papers_to_cite_for_next(c)
    if nxt is not None:
        base = Solution().hIndex(c[:])
        assert nxt >= 1
        assert hindex_after_adding(c, nxt) >= base + 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm thùng, chặn ở `n`, cộng dồn từ lớn xuống nhỏ)

```python
class Solution:
    def hIndex(self, citations: List[int]) -> int:
        n = len(citations)
        bucket = [0] * (n + 1)
        for c in citations:
            bucket[min(c, n)] += 1
        total = 0
        for h in range(n, -1, -1):
            total += bucket[h]
            if total >= h:
                return h
        return 0
```

**Checklist**

- [ ] Hiểu: `h` hợp lệ khi **số bài có `citations ≥ h`** là **`≥ h`**; lấy `h` **lớn nhất**?
- [ ] Nếu sắp xếp: **giảm dần** dùng `c[h − 1] ≥ h`; **tăng dần** dùng `c[n − h] ≥ h`?
- [ ] Vòng lặp `h` bắt đầu từ `n` (có thể `h = n`) và mặc định `0` khi không có `h ≥ 1` nào?
- [ ] Nếu đếm thùng: **chặn** trích dẫn ở `n` (`min(c, n)`) để không vượt chỉ số?
- [ ] Không nhầm h-index với trích dẫn lớn nhất / trung vị?
- [ ] Đã thử: `[0]`, `[1000]`, toàn `0`, toàn `n`, mảng đã sắp, mảng ngược?
- [ ] Đã đối chiếu với brute force đếm trực tiếp mọi `h` trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def hIndex(self, citations: List[int]) -> int`?